The file `RecursiveSet.jsh` provides the class `RecursiveSet`, which is an implementation of mathematical sets that can be nested.  It is discussed in the notebook `00-Introduction-to-Recursive-Set.ipynb`.

In [ ]:
%load ../lib/RecursiveSet.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

# The Usual Suspects
---
There has been a burglary at a jewelry store.  The [usual suspects](https://en.wikipedia.org/wiki/The_Usual_Suspects) have been arrested.  These are
 * Aaron,
 * Bernard, and
 * Caine.
 
Furthermore, the following facts have been established:
 1. It is known that at least one of these suspects is indeed guilty.
 2. If Aaron is guilty, he has exactly one accomplice.
 3. If Bernard is innocent, then Caine is inncocent, too.
 4. If exactly two of the suspects are guilty, then Caine is one of them.
 5. If Caine is innocent, then Aaron is guilty.

It is our task to identify those suspects that are guilty.

Our first task is to define the set of propositional variables:
$$ \mathcal{P} := \{ \texttt{a}, \texttt{b}, \texttt{c} \} $$
The interpretation is that 
 * $\texttt{a}$ is true iff Aaron is guilty,
 * $\texttt{b}$ is true iff Bernard is guilty, and
 * $\texttt{c}$ is true iff Caine is guilty.  

In [ ]:
var P = set("a", "b", "c");
P

Our next task is to translate the facts given above into formulas from propositional logic. 

The statement "*It is known that at least one of these suspects is indeed guilty.*" is translated as follows:
$$ 
\texttt{a} \vee \texttt{b} \vee \texttt{c}. 
$$ 

In [ ]:
var f1 = "a ∨ b ∨ c";

The statement "*If Aaron is guilty, he has exactly one accomplice.*" is harder to translate into propositional logic. The idea is to split this statement into two statements:
* If Aaron is guilty, he has at least one accomplice.</li>
* If Aaron is guilty, he has at most  one accomplice.</li>

These statements can now be translated into the following formulas:

In [ ]:
var f2 = "a → b ∨ c";

In [ ]:
var f3 = "a → ¬(b ∧ c)";

The statement "*If Bernard is innocent, then Caine is inncocent, too.*" is a straightforward implication:

In [ ]:
var f4 = "¬b → ¬c";

The statement "*If exactly two of the suspects are guilty, then Caine is one of them.*" is best translated into propositional logic by asking how this statement could be made false.

This statement is false if and only if
  + two suspects are guilty, 
  + but Caine is innocent.
  
However this is only possible if and only if 
  + *Caine is innocent*,
  + *Aaron and Bernard are guilty*.  
  
Hence we can translate this statement as follows:

In [ ]:
var f5 = "¬(¬c ∧ a ∧ b)";

The statement "*If Caine is innocent, then Aaron is guilty.*" is an implication:

In [ ]:
var f6 = "¬c → a";

We define the list `Fs` of all formulas:

In [ ]:
var Fs = List.of(f1, f2, f3, f4, f5, f6);
Fs

We need to transform the strings <tt>f1</tt> to <tt>f6</tt> into formulas, i.e. into trees of objects.  To this end we load the record classes representing formulas and a parser for propositional formulas.

In [ ]:
%load Formula.jsh
%load PropositionalLogicParser.jsh

Next, we transform all strings into formulas:

In [ ]:
var Gs = Fs.stream().map(f -> parse(f)).toList();
Gs

We are looking for a variable assignment $\mathcal{I}$ that satisfies all formulas in the set <tt>Fs</tt>.  As variable assignments are represented as subsets of the set $\mathcal{P}$ of propositional variables, we can just iterate over all subsets of $\mathcal{P}$.

The function $\texttt{evaluate}(F, I)$ takes a propositional formula $F$ and a propositional variable assignment $I$ and evaluates $F$ using the assignment $I$.  We have discussed the details of this function previously.

In [ ]:
boolean evaluate(Formula f, RecursiveSet<String> I) {
    return switch (f) {
        case Var(var p)            -> I.contains(p);
        case Verum()               -> true;
        case Falsum()              -> false;
        case Not(var g)            -> !evaluate(g, I);
        case And(var g, var h)     -> evaluate(g, I) && evaluate(h, I);
        case Or(var g, var h)      -> evaluate(g, I) || evaluate(h, I);
        case Implies(var g, var h) -> !evaluate(g, I) || evaluate(h, I);
        case Equiv(var g, var h)   -> evaluate(g, I) == evaluate(h, I);
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

The function `allTrue(Gs, I)` takes a list of propositional formulas `Gs`
and a propositional variable assignment `I`.  It returns `true` only if all formulas from `Gs` are
`true` given the variable assignment `I`.

In [ ]:
boolean allTrue(List<Formula> Gs, RecursiveSet<String> I) {
    return Gs.stream().allMatch(f -> evaluate(f, I));
}

The class `RecursiveSet` provides the method `powerset`, which returns the *power set* of a given set, i.e. the set of all subsets of the given set.

In [ ]:
P.powerset()

Next, we compute the set of all variable assignments that render all formulas true:

In [ ]:
var validAssignments = P.powerset().filter(I -> allTrue(Gs, I));
validAssignments

It turns our that there is just one propositional variable assignment that satisfies all formulas from the set <tt>Fs</tt>.  Therefore, the problem has a unique solution: Bernard and Caine are guilty, while Aaron is innocent.